# 02. AGEVector — Vector Mode

pgvector 기반 벡터 검색. `Neo4jVector` / `PGVectorStore`와 동일한 인터페이스.

**사전 준비**
```bash
cd docker && docker compose up -d
pip install "langchain-age[vector]" langchain-openai
```

In [ ]:
import getpass
import os

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")

DSN = "host=localhost port=5433 dbname=langchain_age user=langchain password=langchain"

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

## 1. 기본 벡터 검색

In [ ]:
from langchain_age import AGEVector, DistanceStrategy

store = AGEVector(
    connection_string=DSN,
    embedding_function=embeddings,
    collection_name="demo_vectors",
    distance_strategy=DistanceStrategy.COSINE,
    pre_delete_collection=True,
)

texts = [
    "Apache AGE adds graph query capabilities to PostgreSQL.",
    "pgvector enables fast vector similarity search in PostgreSQL.",
    "LangChain is a framework for building LLM applications.",
    "Neo4j is a native graph database using the Bolt protocol.",
    "PostgreSQL supports JSON, full-text search, and extensions.",
    "RAG combines retrieval and generation for accurate answers.",
    "GraphRAG uses knowledge graphs to enhance LLM context.",
    "Vector embeddings represent text as high-dimensional points.",
]

ids = store.add_texts(texts)
print(f"Added {len(ids)} documents.")

In [ ]:
results = store.similarity_search("graph database PostgreSQL", k=3)
for i, doc in enumerate(results, 1):
    print(f"  {i}. {doc.page_content}")

## 2. 점수와 함께 검색

In [ ]:
# distance (낮을수록 유사)
results = store.similarity_search_with_score("vector search", k=3)
for doc, dist in results:
    print(f"  [dist={dist:.4f}] {doc.page_content}")

print()

# relevance score (0~1, 높을수록 유사)
results = store.similarity_search_with_relevance_scores("vector search", k=3)
for doc, score in results:
    print(f"  [score={score:.4f}] {doc.page_content}")

## 3. 메타데이터 필터링

MongoDB 스타일 필터: `$eq`, `$ne`, `$lt`, `$gt`, `$in`, `$between`, `$like`, `$and`, `$or`

In [ ]:
store.add_texts(
    ["Python is great for data science.", "TypeScript is popular for web dev.", "Rust is fast and safe."],
    metadatas=[
        {"lang": "python", "year": "2024", "category": "backend"},
        {"lang": "typescript", "year": "2024", "category": "frontend"},
        {"lang": "rust", "year": "2024", "category": "systems"},
    ],
)

# 단순 필터
print("=== lang=python ===")
for doc in store.similarity_search("programming", k=3, filter={"lang": "python"}):
    print(f"  {doc.page_content}")

# $or 필터
print("\n=== lang in [python, rust] ===")
for doc in store.similarity_search("programming", k=3, filter={"lang": {"$in": ["python", "rust"]}}):
    print(f"  {doc.page_content}")

## 4. 하이브리드 검색 (Vector + Full-text via RRF)

In [ ]:
from langchain_age import SearchType

hybrid_store = AGEVector(
    connection_string=DSN,
    embedding_function=embeddings,
    collection_name="demo_vectors",
    search_type=SearchType.HYBRID,
)

results = hybrid_store.similarity_search("PostgreSQL graph extensions", k=3)
for i, doc in enumerate(results, 1):
    print(f"  {i}. {doc.page_content}")

## 5. MMR (Maximal Marginal Relevance)

In [ ]:
results = store.max_marginal_relevance_search(
    "database technology", k=3, fetch_k=8, lambda_mult=0.5,
)
for i, doc in enumerate(results, 1):
    print(f"  {i}. {doc.page_content}")

## 6. HNSW 인덱스

In [ ]:
store.create_hnsw_index(m=16, ef_construction=64)
print("HNSW index created.")

results = store.similarity_search("graph database", k=2)
for doc in results:
    print(f"  {doc.page_content}")

## 7. LangChain Retriever

In [ ]:
retriever = store.as_retriever(search_kwargs={"k": 3})
docs = retriever.invoke("What is RAG?")
for doc in docs:
    print(f"  {doc.page_content}")

## 8. 정리

In [ ]:
store.drop_index()
store._drop_table()
store.close()
print("Done.")